# How to read an ORC file with Python

Read an ORC file's footer, load it into pandas with Rugo, select columns, see how nested types come across, and convert to Parquet to query with Opteryx.

This notebook accompanies [How to read an ORC file with Python](https://docs.opteryx.app/docs/guides/read-an-orc-file-with-python) on docs.opteryx.app.

In [ ]:
%pip install -q rugo opteryx-core pandas pyarrow

## Get the data

`space_missions.orc`: 4,630 orbital launch attempts, with each rocket as a nested struct.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://docs.opteryx.app/data/space_missions.orc", "space_missions.orc")

## Read the footer first

The footer is at the end of the file, so nothing is decoded.

In [ ]:
from rugo.orc import read_metadata, read_orc

meta = read_metadata("space_missions.orc")
print(meta.compression, meta.num_rows, meta.stripes)
for column in meta.columns:
    print(column["name"], column["kind"])

## Load an ORC file into pandas

`read_orc` yields one morsel per stripe. Types come from the footer; the nested `Rocket` struct arrives as JSON text.

In [ ]:
import pyarrow

df = pyarrow.concat_tables(morsel.to_arrow() for morsel in read_orc("space_missions.orc")).to_pandas()

df.dtypes

In [ ]:
df.head()

## Read only the columns you need

Streams for the columns you leave out are skipped rather than decoded. `columns` takes top-level names only.

In [ ]:
df = pyarrow.concat_tables(
    morsel.to_arrow()
    for morsel in read_orc("space_missions.orc", columns=["Company", "Mission", "Rocket"])
).to_pandas()

df.head()

## Take a field out of a nested struct

A dotted name such as `Rocket.Name` is refused, so read `Rocket` and parse the JSON text.

In [ ]:
import json

df["Rocket_Name"] = df["Rocket"].map(lambda r: json.loads(r)["Name"] if r else None)
df[["Mission", "Rocket_Name"]].head()

## Filtering rows

There is no `predicates` argument, so filter after reading.

In [ ]:
spacex = df[df["Company"] == "SpaceX"]
len(spacex)

## Query with SQL

Opteryx reads ORC directly with `READ_ORC`, with `->>` reaching into the nested struct.

In [ ]:
import opteryx

morsels = opteryx.session().execute_to_morsels("""
    SELECT Company, COUNT(*) AS launches
      FROM READ_ORC('space_missions.orc')
     WHERE Rocket->>'Status' = 'Active'
     GROUP BY Company
     ORDER BY launches DESC
     LIMIT 5
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

## Convert to Parquet for repeated queries

If you will query the data more than once, convert it once with Rugo and query the Parquet copy. Rugo writes one row group per stripe, with the types carried across.

In [ ]:
from rugo import parquet

with open("space_missions.parquet", "wb") as f:
    with parquet.open_parquet_writer(f.write) as writer:
        for morsel in read_orc("space_missions.orc"):
            writer.write_row_group(morsel)

!rugo schema space_missions.parquet

In [ ]:
morsels = opteryx.session().execute_to_morsels("""
    SELECT Company, COUNT(*) AS launches
      FROM READ_PARQUET('space_missions.parquet')
     WHERE Rocket->>'Status' = 'Active'
     GROUP BY Company
     ORDER BY launches DESC
     LIMIT 5
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()